In [1]:
"""
Local Curvature Analysis of Cosmic Ray Time Series - v2
=========================================================
Fixes:
  - Clean separation of OMNI correlations and cross-station correlations (no NaN mixing)
  - Optimized for daily resolution (FR only mode)
  - Better summary tables

Changes from v1:
  - compute_correlations and compute_cross_station_correlation output unified columns
  - Added 'Type' column to distinguish OMNI vs cross-station correlations
  - Summary printed as two clean separate tables
  - Added progress tracking for large daily VGs

Author: D. Sierra-Porta
"""

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import linprog
from scipy.stats import pearsonr, spearmanr, kendalltau
import warnings
import time
import os

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (16, 3),
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})


# ============================================================
# SECTION 1: DATA LOADING AND PREPARATION
# ============================================================

def load_and_prepare_data(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(filepath, index_col='DATETIME', parse_dates=True)
    elif ext in ['.pkl', '.pickle']:
        df = pd.read_pickle(filepath)
    elif ext == '.parquet':
        df = pd.read_parquet(filepath)
    else:
        df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    
    col_map = {
        'Scalar B, nT': 'B',
        'SW Plasma Temperature, K': 'Temp',
        'SW Proton Density, N/cm^3': 'Density',
        'SW Plasma Speed, km/s': 'Speed',
        'R (Sunspot No.)': 'SSN',
        'Dst-index, nT': 'Dst'
    }
    df = df.rename(columns=col_map)
    
    print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print(f"Cycles present: {sorted(df['Cycle'].dropna().unique().astype(int))}")
    print(f"\nMissing data (%):")
    for col in df.columns:
        pct = df[col].isna().sum() / len(df) * 100
        print(f"  {col:30s}: {pct:.1f}%")
    
    return df


def resample_data(df, freq='W'):
    if freq == 'D':
        return df.copy()
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Cycle' in numeric_cols:
        numeric_cols.remove('Cycle')
    
    df_resampled = df[numeric_cols].resample(freq).mean()
    
    if 'Cycle' in df.columns:
        df_resampled['Cycle'] = df['Cycle'].resample(freq).apply(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan
        )
    
    return df_resampled


def get_cycle_data(df, cycle, cr_col='JUNG'):
    mask = df['Cycle'] == cycle
    cycle_data = df.loc[mask].copy()
    
    if cycle_data.empty:
        print(f"  WARNING: No data for cycle {cycle}")
        return None, None
    
    series = cycle_data[cr_col].interpolate(method='linear', limit=5)
    series = series.dropna()
    
    if len(series) < 20:
        print(f"  WARNING: Cycle {cycle}, {cr_col} has only {len(series)} valid points")
        return None, None
    
    return series, series.index


# ============================================================
# SECTION 2: CURVATURE IMPLEMENTATIONS
# ============================================================

def forman_ricci_local(G):
    """
    Forman-Ricci curvature for unweighted graphs.
    F(e=(u,v)) = |common_neighbors| + 2 - |symmetric_diff_neighbors|
    Node curvature = mean of adjacent edge curvatures.
    """
    edge_curvatures = {}
    
    for u, v in G.edges():
        neighbors_u = set(G.neighbors(u)) - {v}
        neighbors_v = set(G.neighbors(v)) - {u}
        common = len(neighbors_u & neighbors_v)
        non_common = len(neighbors_u.symmetric_difference(neighbors_v))
        edge_curvatures[(u, v)] = common + 2 - non_common
    
    node_curvatures = {}
    for n in G.nodes():
        adj_curvs = []
        for u, v in G.edges(n):
            key = (u, v) if (u, v) in edge_curvatures else (v, u)
            if key in edge_curvatures:
                adj_curvs.append(edge_curvatures[key])
        node_curvatures[n] = np.mean(adj_curvs) if adj_curvs else 0.0
    
    return edge_curvatures, node_curvatures


def _wasserstein_1d(mu, nu, cost_matrix):
    n = len(mu)
    m = len(nu)
    c = cost_matrix.flatten()
    A_eq = np.zeros((n + m, n * m))
    for i in range(n):
        A_eq[i, i * m:(i + 1) * m] = 1
    for j in range(m):
        A_eq[n + j, j::m] = 1
    b_eq = np.concatenate([mu, nu])
    result = linprog(c, A_eq=A_eq, b_eq=b_eq,
                     bounds=[(0, None)] * (n * m),
                     method='highs', options={'presolve': True})
    return result.fun if result.success else np.nan


def ollivier_ricci_local(G, alpha=0.5, max_edges=600, verbose=True):
    """Ollivier-Ricci curvature via optimal transport."""
    nodes = list(G.nodes())
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}
    
    if verbose:
        print(f"  [OR] Computing shortest paths ({n} nodes)...")
    dist_dict = dict(nx.all_pairs_shortest_path_length(G))
    
    dist_matrix = np.full((n, n), np.inf)
    for u in nodes:
        for v, d in dist_dict[u].items():
            dist_matrix[node_idx[u]][node_idx[v]] = d
    
    edges = list(G.edges())
    if len(edges) > max_edges:
        if verbose:
            print(f"  [OR] Sampling {max_edges}/{len(edges)} edges...")
        sample_idx = np.random.choice(len(edges), max_edges, replace=False)
        edges_to_compute = [edges[i] for i in sample_idx]
    else:
        edges_to_compute = edges
    
    if verbose:
        print(f"  [OR] Computing curvature for {len(edges_to_compute)} edges...")
    
    edge_curvatures = {}
    
    for idx, (u, v) in enumerate(edges_to_compute):
        if verbose and idx % 500 == 0 and idx > 0:
            print(f"    ... {idx}/{len(edges_to_compute)}")
        
        neighbors_u = list(G.neighbors(u))
        support_u = [u] + neighbors_u
        mu = np.zeros(len(support_u))
        mu[0] = alpha
        if neighbors_u:
            mu[1:] = (1 - alpha) / len(neighbors_u)
        else:
            mu[0] = 1.0
        
        neighbors_v = list(G.neighbors(v))
        support_v = [v] + neighbors_v
        nu = np.zeros(len(support_v))
        nu[0] = alpha
        if neighbors_v:
            nu[1:] = (1 - alpha) / len(neighbors_v)
        else:
            nu[0] = 1.0
        
        cost = np.zeros((len(support_u), len(support_v)))
        for i, su in enumerate(support_u):
            for j, sv in enumerate(support_v):
                cost[i, j] = dist_matrix[node_idx[su]][node_idx[sv]]
        
        W = _wasserstein_1d(mu, nu, cost)
        d_uv = dist_matrix[node_idx[u]][node_idx[v]]
        
        if d_uv > 0 and not np.isnan(W):
            edge_curvatures[(u, v)] = 1.0 - W / d_uv
        else:
            edge_curvatures[(u, v)] = 0.0
    
    node_curvatures = {}
    for nd in G.nodes():
        adj_curv = []
        for (u, v), c in edge_curvatures.items():
            if u == nd or v == nd:
                adj_curv.append(c)
        node_curvatures[nd] = np.mean(adj_curv) if adj_curv else np.nan
    
    return edge_curvatures, node_curvatures


def ricci_flow_local(G, alpha=0.5, iterations=20, verbose=True):
    """Discrete Ricci flow."""
    G_flow = G.copy()
    for u, v in G_flow.edges():
        G_flow[u][v]['weight'] = 1.0
    
    if verbose:
        print(f"  [RF] Running Ricci flow ({iterations} iterations, {G.number_of_nodes()} nodes)...")
    
    for it in range(iterations):
        if verbose and it % 5 == 0:
            print(f"    ... iteration {it}/{iterations}")
        
        dist_dict = dict(nx.all_pairs_dijkstra_path_length(G_flow, weight='weight'))
        
        for u, v in G_flow.edges():
            neighbors_u = list(G_flow.neighbors(u))
            neighbors_v = list(G_flow.neighbors(v))
            
            support_u = [u] + neighbors_u
            mu = np.zeros(len(support_u))
            mu[0] = alpha
            if neighbors_u:
                mu[1:] = (1 - alpha) / len(neighbors_u)
            else:
                mu[0] = 1.0
            
            support_v = [v] + neighbors_v
            nu = np.zeros(len(support_v))
            nu[0] = alpha
            if neighbors_v:
                nu[1:] = (1 - alpha) / len(neighbors_v)
            else:
                nu[0] = 1.0
            
            cost = np.zeros((len(support_u), len(support_v)))
            for i, su in enumerate(support_u):
                for j, sv in enumerate(support_v):
                    cost[i, j] = dist_dict.get(su, {}).get(sv, 1e6)
            
            W = _wasserstein_1d(mu, nu, cost)
            d_uv = dist_dict.get(u, {}).get(v, 1.0)
            kappa = (1.0 - W / d_uv) if (d_uv > 0 and not np.isnan(W)) else 0.0
            G_flow[u][v]['weight'] = max((1 - kappa) * d_uv, 1e-6)
    
    edge_flow = {(u, v): G_flow[u][v]['weight'] for u, v in G_flow.edges()}
    node_flow = {}
    for nd in G_flow.nodes():
        adj_w = [G_flow[u][v]['weight'] for u, v in G_flow.edges(nd)]
        node_flow[nd] = np.mean(adj_w) if adj_w else np.nan
    
    return edge_flow, node_flow


# ============================================================
# SECTION 3: CURVATURE TIME SERIES BUILDER
# ============================================================

def build_curvature_timeseries(series, dates, compute_or=True, compute_rf=False,
                                alpha=0.5, rf_iterations=20, rf_max_nodes=80,
                                or_max_edges=600, verbose=True):
    values = np.array(series)
    n = len(values)
    
    if verbose:
        print(f"  Building VG ({n} nodes)...")
    
    t0 = time.time()
    G = nx.visibility_graph(values)
    t_vg = time.time() - t0
    
    if verbose:
        print(f"  VG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ({t_vg:.1f}s)")
    
    result = {
        'dates': dates,
        'values': values,
        'G': G,
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges()
    }
    
    # --- Forman-Ricci (always computed, very fast) ---
    t0 = time.time()
    fr_edge, fr_node = forman_ricci_local(G)
    t_fr = time.time() - t0
    
    result['FR_edge'] = fr_edge
    result['FR_node'] = np.array([fr_node.get(i, np.nan) for i in range(n)])
    
    if verbose:
        print(f"  FR: mean={np.nanmean(result['FR_node']):.3f}, "
              f"std={np.nanstd(result['FR_node']):.3f} ({t_fr:.1f}s)")
    
    # --- Ollivier-Ricci ---
    if compute_or:
        t0 = time.time()
        or_edge, or_node = ollivier_ricci_local(G, alpha=alpha,
                                                  max_edges=or_max_edges,
                                                  verbose=verbose)
        t_or = time.time() - t0
        result['OR_edge'] = or_edge
        result['OR_node'] = np.array([or_node.get(i, np.nan) for i in range(n)])
        if verbose:
            print(f"  OR: mean={np.nanmean(result['OR_node']):.3f}, "
                  f"std={np.nanstd(result['OR_node']):.3f} ({t_or:.1f}s)")
    else:
        result['OR_edge'] = None
        result['OR_node'] = None
    
    # --- Ricci Flow ---
    if compute_rf and n <= rf_max_nodes:
        t0 = time.time()
        rf_edge, rf_node = ricci_flow_local(G, alpha=alpha,
                                             iterations=rf_iterations, verbose=verbose)
        t_rf = time.time() - t0
        result['RF_edge'] = rf_edge
        result['RF_node'] = np.array([rf_node.get(i, np.nan) for i in range(n)])
        if verbose:
            print(f"  RF: mean={np.nanmean(result['RF_node']):.3f}, "
                  f"std={np.nanstd(result['RF_node']):.3f} ({t_rf:.1f}s)")
    else:
        if compute_rf and n > rf_max_nodes and verbose:
            print(f"  RF: SKIPPED (n={n} > rf_max_nodes={rf_max_nodes})")
        result['RF_edge'] = None
        result['RF_node'] = None
    
    return result


# ============================================================
# SECTION 4: CORRELATION ANALYSIS (FIXED - no NaN mixing)
# ============================================================

def compute_omni_correlations(curv_series, omni_df, curv_dates,
                               curv_label='FR', station='', cycle=0):
    """
    Correlate curvature time series with OMNI space weather variables.
    Returns a clean DataFrame with consistent columns.
    """
    omni_vars = ['B', 'Temp', 'Density', 'Speed', 'SSN', 'Dst']
    omni_labels = ['Scalar B (nT)', 'SW Temperature (K)', 'Proton Density (N/cm³)',
                   'SW Speed (km/s)', 'Sunspot Number', 'Dst Index (nT)']
    
    records = []
    
    for var, label in zip(omni_vars, omni_labels):
        if var not in omni_df.columns:
            continue
        
        omni_at_dates = omni_df[var].reindex(curv_dates)
        mask = ~(np.isnan(curv_series) | np.isnan(omni_at_dates.values))
        
        if mask.sum() < 10:
            continue
        
        x = omni_at_dates.values[mask]
        y = curv_series[mask]
        
        r_p, p_p = pearsonr(x, y)
        r_s, p_s = spearmanr(x, y)
        tau, p_tau = kendalltau(x, y)
        
        records.append({
            'Cycle': cycle,
            'Station': station,
            'Curvature': curv_label,
            'Variable': label,
            'N': int(mask.sum()),
            'Pearson_r': round(r_p, 4),
            'Pearson_p': p_p,
            'Spearman_rho': round(r_s, 4),
            'Spearman_p': p_s,
            'Kendall_tau': round(tau, 4),
            'Kendall_p': p_tau
        })
    
    return pd.DataFrame(records)


def compute_cross_station_correlation(result_a, result_b,
                                       label_a='JUNG', label_b='MOSC', cycle=0):
    """
    Correlate curvature time series between two stations.
    Returns a clean DataFrame with consistent columns.
    """
    records = []
    
    for metric in ['FR_node', 'OR_node', 'RF_node']:
        if result_a[metric] is None or result_b[metric] is None:
            continue
        
        dates_a = result_a['dates']
        dates_b = result_b['dates']
        common_dates = dates_a.intersection(dates_b)
        
        if len(common_dates) < 10:
            continue
        
        idx_a = [list(dates_a).index(d) for d in common_dates if d in dates_a]
        idx_b = [list(dates_b).index(d) for d in common_dates if d in dates_b]
        
        min_len = min(len(idx_a), len(idx_b))
        a = result_a[metric][idx_a[:min_len]]
        b = result_b[metric][idx_b[:min_len]]
        
        mask = ~(np.isnan(a) | np.isnan(b))
        if mask.sum() < 10:
            continue
        
        r_p, p_p = pearsonr(a[mask], b[mask])
        r_s, p_s = spearmanr(a[mask], b[mask])
        tau, p_tau = kendalltau(a[mask], b[mask])
        
        metric_name = metric.replace('_node', '')
        
        records.append({
            'Cycle': cycle,
            'Stations': f'{label_a} vs {label_b}',
            'Curvature': metric_name,
            'N': int(mask.sum()),
            'Pearson_r': round(r_p, 4),
            'Pearson_p': p_p,
            'Spearman_rho': round(r_s, 4),
            'Spearman_p': p_s,
            'Kendall_tau': round(tau, 4),
            'Kendall_p': p_tau
        })
    
    return pd.DataFrame(records)


# ============================================================
# SECTION 5: ANOMALY DETECTION
# ============================================================

def detect_curvature_anomalies(curv_series, dates, sigma=2.0, metric_name='FR'):
    mean = np.nanmean(curv_series)
    std = np.nanstd(curv_series)
    
    threshold_low = mean - sigma * std
    threshold_high = mean + sigma * std
    
    mask_low = curv_series < threshold_low
    mask_high = curv_series > threshold_high
    
    anomalies_low = pd.DataFrame({
        'date': dates[mask_low],
        'curvature': curv_series[mask_low],
        'type': 'low',
        'metric': metric_name
    })
    
    anomalies_high = pd.DataFrame({
        'date': dates[mask_high],
        'curvature': curv_series[mask_high],
        'type': 'high',
        'metric': metric_name
    })
    
    anomalies = pd.concat([anomalies_low, anomalies_high]).sort_values('date')
    return anomalies, threshold_low, threshold_high


# ============================================================
# SECTION 6: VISUALIZATION
# ============================================================

def plot_curvature_timeseries(results_dict, station_labels, save_prefix='fig'):
    n_stations = len(station_labels)
    has_or = any(results_dict[s]['OR_node'] is not None for s in station_labels)
    n_panels = n_stations + 1 + (1 if has_or else 0)
    
    fig, axes = plt.subplots(n_panels, 1, figsize=(16, 2.5 * n_panels), sharex=True)
    #fig.suptitle('Local Curvature Dynamics: Cosmic Ray Time Series',
                 #fontsize=14, fontweight='bold', y=0.98)
    
    colors = ['#1F2CB4', '#77B41F', '#d62728', '#9467bd', '#8c564b']
    
    for i, label in enumerate(station_labels):
        res = results_dict[label]
        axes[i].plot(res['dates'], res['values'], color=colors[i % len(colors)],
                     linewidth=1.2, alpha=0.8)
        axes[i].set_ylabel(f'CR Counts ({label})')
        axes[i].set_title(f'({chr(97+i)}) {label} Cosmic Ray Intensity', loc='left', fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim(res['dates'].min(),res['dates'].max())
    
    ax_fr = axes[n_stations]
    for i, label in enumerate(station_labels):
        res = results_dict[label]
        ax_fr.plot(res['dates'], res['FR_node'], color=colors[i % len(colors)],
                   linewidth=1.2, alpha=0.7, label=f'{label} (FR)')
    ax_fr.axhline(0, color='gray', linestyle=':', linewidth=0.5)
    ax_fr.set_ylabel('Forman-Ricci Curvature')
    ax_fr.set_title(f'({chr(97+n_stations)}) Local Forman-Ricci Curvature', loc='left', fontsize=12)
    ax_fr.legend(loc='lower right', fontsize=9)
    ax_fr.grid(True, alpha=0.3)
    ax_fr.set_xlim(res['dates'].min(),res['dates'].max())
    
    if has_or:
        ax_or = axes[n_stations + 1]
        for i, label in enumerate(station_labels):
            res = results_dict[label]
            if res['OR_node'] is not None:
                ax_or.plot(res['dates'], res['OR_node'], color=colors[i % len(colors)],
                           linewidth=0.7, alpha=0.7, label=f'{label} (OR)')
        ax_or.axhline(0, color='gray', linestyle=':', linewidth=0.5)
        ax_or.set_ylabel('Ollivier-Ricci\nCurvature')
        ax_or.set_title(f'({chr(97+n_stations+1)}) Local Ollivier-Ricci Curvature', loc='left', fontsize=11)
        ax_or.legend(loc='upper right', fontsize=9)
        ax_or.grid(True, alpha=0.3)
    
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_timeseries.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_timeseries.png")


def plot_curvature_vs_omni(result, omni_df, station_label, curv_metric='FR_node',
                            save_prefix='fig'):
    omni_vars = ['B', 'Speed', 'Dst', 'SSN', 'Density', 'Temp']
    omni_labels = ['Scalar B (nT)', 'SW Speed (km/s)', 'Dst (nT)',
                   'Sunspot No.', 'Proton Density', 'SW Temp (K)']
    
    available = [(v, l) for v, l in zip(omni_vars, omni_labels) if v in omni_df.columns]
    n_plots = len(available)
    ncols = 3
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    metric_label = curv_metric.replace('_node', '')
    fig.suptitle(f'{metric_label} Curvature ({station_label}) vs Space Weather Variables',
                 fontsize=13, fontweight='bold')
    
    curv = result[curv_metric]
    dates = result['dates']
    
    for idx, (var, label) in enumerate(available):
        ax = axes[idx]
        omni_at_dates = omni_df[var].reindex(dates)
        mask = ~(np.isnan(curv) | np.isnan(omni_at_dates.values))
        
        if mask.sum() < 5:
            ax.set_visible(False)
            continue
        
        x = omni_at_dates.values[mask]
        y = curv[mask]
        
        ax.scatter(x, y, s=10, alpha=0.4, c='steelblue', edgecolors='none')
        
        r_p, p_p = pearsonr(x, y)
        r_s, p_s = spearmanr(x, y)
        
        z = np.polyfit(x, y, 1)
        p_fit = np.poly1d(z)
        x_sorted = np.sort(x)
        ax.plot(x_sorted, p_fit(x_sorted), 'r-', linewidth=1.5, alpha=0.7)
        
        sig_p = '***' if p_p < 0.001 else ('**' if p_p < 0.01 else ('*' if p_p < 0.05 else 'ns'))
        ax.set_title(f'{label}\nr={r_p:.3f} {sig_p}, ρ={r_s:.3f}', fontsize=10)
        ax.set_xlabel(label, fontsize=9)
        ax.set_ylabel(f'{metric_label} Curvature', fontsize=9)
        ax.grid(True, alpha=0.2)
    
    for idx in range(len(available), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_vs_omni_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_vs_omni_{station_label}.png")


def plot_anomaly_detection(result, omni_df, station_label, sigma=2.0, save_prefix='fig'):
    fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
    fig.suptitle(f'Curvature Anomaly Detection ({station_label})',
                 fontsize=13, fontweight='bold', y=0.96)
    
    dates = result['dates']
    
    axes[0].plot(dates, result['values'], 'b-', linewidth=0.8)
    axes[0].set_ylabel('CR Counts')
    axes[0].set_title(f'(a) {station_label} Cosmic Ray Intensity', loc='left')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(dates.min(),dates.max())
    
    fr = result['FR_node']
    anomalies, thr_low, thr_high = detect_curvature_anomalies(fr, dates, sigma=sigma)
    
    axes[1].plot(dates, fr, 'steelblue', linewidth=1.2)
    axes[1].axhline(np.nanmean(fr), color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    axes[1].axhline(thr_low, color='red', linestyle='--', linewidth=0.8,
                     label=f'$\\mu$ - {sigma}$\\sigma$ = {thr_low:.2f}')
    axes[1].axhline(thr_high, color='orange', linestyle='--', linewidth=0.8,
                     label=f'$\\mu$ + {sigma}$\\sigma$ = {thr_high:.2f}')
    
    anom_low = anomalies[anomalies['type'] == 'low']
    anom_high = anomalies[anomalies['type'] == 'high']
    
    if not anom_low.empty:
        anom_idx = [list(dates).index(d) for d in anom_low['date'] if d in dates]
        axes[1].scatter(dates[anom_idx], fr[anom_idx], c='red', s=35, zorder=5,
                       label=f'Low anomalies ({len(anom_low)})')
    if not anom_high.empty:
        anom_idx = [list(dates).index(d) for d in anom_high['date'] if d in dates]
        axes[1].scatter(dates[anom_idx], fr[anom_idx], c='orange', s=35, zorder=5,
                       label=f'High anomalies ({len(anom_high)})')
    
    axes[1].set_ylabel('Forman-Ricci Curvature')
    axes[1].set_title('(b) Local Forman-Ricci with Anomaly Detection', loc='left')
    axes[1].legend(loc='lower right', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(dates.min(),dates.max())

    if 'Dst' in omni_df.columns:
        dst_at_dates = omni_df['Dst'].reindex(dates)
        axes[2].plot(dates, dst_at_dates, 'darkgreen', linewidth=1.2)
        axes[2].axhline(0, color='gray', linestyle=':', linewidth=0.8)
        axes[2].set_ylabel('Dst Index (nT)')
        axes[2].set_title('(c) Dst Index (Geomagnetic Storm Indicator)', loc='left')
        axes[2].grid(True, alpha=0.3)
    
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[2].set_xlim(dates.min(),dates.max())
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_anomaly_detection_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_anomaly_detection_{station_label}.png")
    
    return anomalies


def plot_curvature_distribution(results_dict, station_labels, save_prefix='fig'):
    n_stations = len(station_labels)
    fig, axes = plt.subplots(1, n_stations, figsize=(6 * n_stations, 5))
    if n_stations == 1:
        axes = [axes]
    
    fig.suptitle('Distribution of Local Forman-Ricci Curvature', fontsize=13, fontweight='bold')
    
    for i, label in enumerate(station_labels):
        fr = results_dict[label]['FR_node']
        fr_clean = fr[~np.isnan(fr)]
        axes[i].hist(fr_clean, bins=30, color='steelblue', alpha=0.7, edgecolor='white', density=True)
        axes[i].axvline(np.mean(fr_clean), color='red', linestyle='--', linewidth=1.5,
                        label=f'$\\mu$ = {np.mean(fr_clean):.2f}')
        axes[i].axvline(np.median(fr_clean), color='darkorange', linestyle='--', linewidth=1.5,
                        label=f'median = {np.median(fr_clean):.2f}')
        axes[i].set_xlabel('Forman-Ricci Curvature')
        axes[i].set_ylabel('Density')
        axes[i].set_title(f'{label} (N={len(fr_clean)})')
        axes[i].legend(fontsize=9)
        axes[i].grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_curvature_distribution.png')
    plt.close()
    print(f"  Saved: {save_prefix}_curvature_distribution.png")


def plot_ricci_flow_detail(result, station_label, save_prefix='fig'):
    if result['RF_node'] is None:
        return
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f'Ricci Flow Analysis ({station_label})', fontsize=13, fontweight='bold')
    dates = result['dates']
    
    axes[0].plot(dates, result['values'], 'b-o', markersize=3, linewidth=0.8)
    axes[0].set_ylabel('CR Counts')
    axes[0].set_title('(a) CR Counts', loc='left')
    axes[0].grid(True, alpha=0.3)
    
    rf = result['RF_node']
    colors_bar = ['coral' if v > np.nanmean(rf) else 'steelblue' for v in rf]
    axes[1].bar(dates, rf, width=5, color=colors_bar, alpha=0.7, edgecolor='none')
    axes[1].axhline(np.nanmean(rf), color='gray', linestyle='--', linewidth=0.8,
                     label=f'Mean = {np.nanmean(rf):.3f}')
    axes[1].set_ylabel('Ricci Flow\n(mean edge weight)')
    axes[1].set_title('(b) Local Ricci Flow per Node', loc='left')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_ricci_flow_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_ricci_flow_{station_label}.png")


# ============================================================
# SECTION 7: MAIN EXECUTION (FIXED)
# ============================================================

def run_full_analysis(filepath, cr_stations=['JUNG', 'MOSC'],
                      cycles=None, freq='W',
                      compute_or=True, compute_rf=False,
                      alpha=0.5, rf_iterations=20, rf_max_nodes=80,
                      or_max_edges=600, sigma_anomaly=2.0,
                      output_dir='./output'):
    """
    Run the complete local curvature analysis pipeline.
    
    Returns
    -------
    all_results : dict  {cycle: {station: result_dict}}
    df_omni_corr : pd.DataFrame  (OMNI correlations, clean)
    df_cross_corr : pd.DataFrame  (cross-station correlations, clean)
    df_anom : pd.DataFrame  (detected anomalies)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("=" * 70)
    print("LOCAL CURVATURE ANALYSIS OF COSMIC RAY TIME SERIES (v2)")
    print("=" * 70)
    
    # --- Load data ---
    print("\n[1] Loading data...")
    df = load_and_prepare_data(filepath)
    
    if cycles is None:
        cycles = sorted(df['Cycle'].dropna().unique().astype(int))
    
    print(f"\nCycles to analyze: {cycles}")
    print(f"Resampling frequency: {freq}")
    print(f"Stations: {cr_stations}")
    print(f"Compute OR: {compute_or} | Compute RF: {compute_rf}")
    
    # --- Resample ---
    print(f"\n[2] Resampling to {freq}...")
    df_resampled = resample_data(df, freq=freq)
    print(f"  Resampled shape: {df_resampled.shape}")
    
    # --- Process ---
    all_results = {}
    all_omni_corr = []      # OMNI correlations (separate list)
    all_cross_corr = []     # Cross-station correlations (separate list)
    all_anomalies = []
    
    for cycle in cycles:
        print(f"\n{'='*50}")
        print(f"  CYCLE {cycle}")
        print(f"{'='*50}")
        
        cycle_results = {}
        
        for station in cr_stations:
            if station not in df_resampled.columns:
                print(f"  WARNING: {station} not in data, skipping.")
                continue
            
            print(f"\n  --- {station} (Cycle {cycle}) ---")
            
            series, dates = get_cycle_data(df_resampled, cycle, cr_col=station)
            if series is None:
                continue
            
            print(f"  Series length: {len(series)} "
                  f"({dates[0].strftime('%Y-%m-%d')} to {dates[-1].strftime('%Y-%m-%d')})")
            
            result = build_curvature_timeseries(
                series.values, dates,
                compute_or=compute_or,
                compute_rf=compute_rf,
                alpha=alpha,
                rf_iterations=rf_iterations,
                rf_max_nodes=rf_max_nodes,
                or_max_edges=or_max_edges,
                verbose=True
            )
            
            cycle_results[station] = result
            
            # OMNI correlations
            print(f"\n  Computing correlations with OMNI variables...")
            corr_fr = compute_omni_correlations(
                result['FR_node'], df_resampled, dates,
                curv_label='FR', station=station, cycle=cycle
            )
            all_omni_corr.append(corr_fr)
            
            if result['OR_node'] is not None:
                corr_or = compute_omni_correlations(
                    result['OR_node'], df_resampled, dates,
                    curv_label='OR', station=station, cycle=cycle
                )
                all_omni_corr.append(corr_or)
            
            # Anomaly detection
            anomalies, _, _ = detect_curvature_anomalies(
                result['FR_node'], dates, sigma=sigma_anomaly
            )
            anomalies['Station'] = station
            anomalies['Cycle'] = cycle
            all_anomalies.append(anomalies)
            
            print(f"  Anomalies detected: {len(anomalies)} "
                  f"({len(anomalies[anomalies['type']=='low'])} low, "
                  f"{len(anomalies[anomalies['type']=='high'])} high)")
        
        all_results[cycle] = cycle_results
        
        # --- Cross-station correlations (separate) ---
        stations_in_cycle = list(cycle_results.keys())
        if len(stations_in_cycle) >= 2:
            cross_corr = compute_cross_station_correlation(
                cycle_results[stations_in_cycle[0]],
                cycle_results[stations_in_cycle[1]],
                stations_in_cycle[0], stations_in_cycle[1],
                cycle=cycle
            )
            if not cross_corr.empty:
                all_cross_corr.append(cross_corr)
        
        # --- Plots ---
        if cycle_results:
            print(f"\n  Generating plots for cycle {cycle}...")
            cycle_prefix = os.path.join(output_dir, f'cycle{cycle}')
            
            plot_curvature_timeseries(cycle_results, list(cycle_results.keys()),
                                      save_prefix=cycle_prefix)
            
            for station in cycle_results:
                plot_curvature_vs_omni(cycle_results[station], df_resampled,
                                       station, save_prefix=cycle_prefix)
                plot_anomaly_detection(cycle_results[station], df_resampled,
                                       station, sigma=sigma_anomaly,
                                       save_prefix=cycle_prefix)
                if cycle_results[station]['RF_node'] is not None:
                    plot_ricci_flow_detail(cycle_results[station], station,
                                           save_prefix=cycle_prefix)
            
            plot_curvature_distribution(cycle_results, list(cycle_results.keys()),
                                        save_prefix=cycle_prefix)
    
    # ============================================================
    # COMPILE AND PRINT CLEAN RESULTS
    # ============================================================
    
    print("\n" + "=" * 70)
    print("RESULTS: OMNI Correlations")
    print("=" * 70)
    
    if all_omni_corr:
        df_omni_corr = pd.concat(all_omni_corr, ignore_index=True)
        df_omni_corr.to_csv(os.path.join(output_dir, 'omni_correlations.csv'), index=False)
        
        # Pretty print
        display_cols = ['Cycle', 'Station', 'Curvature', 'Variable', 'N',
                        'Pearson_r', 'Pearson_p', 'Spearman_rho', 'Spearman_p']
        print(df_omni_corr[display_cols].to_string(index=False))
    else:
        df_omni_corr = pd.DataFrame()
    
    print("\n" + "=" * 70)
    print("RESULTS: Cross-Station Correlations")
    print("=" * 70)
    
    if all_cross_corr:
        df_cross_corr = pd.concat(all_cross_corr, ignore_index=True)
        df_cross_corr.to_csv(os.path.join(output_dir, 'cross_station_correlations.csv'), index=False)
        print(df_cross_corr.to_string(index=False))
    else:
        df_cross_corr = pd.DataFrame()
    
    print("\n" + "=" * 70)
    print("RESULTS: Anomalies Detected")
    print("=" * 70)
    
    if all_anomalies:
        df_anom = pd.concat(all_anomalies, ignore_index=True)
        df_anom.to_csv(os.path.join(output_dir, 'anomalies.csv'), index=False)
        print(f"Total anomalies: {len(df_anom)}")
        print(f"\nPer cycle/station:")
        print(df_anom.groupby(['Cycle', 'Station', 'type']).size().to_string())
    else:
        df_anom = pd.DataFrame()
    
    print(f"\nAll outputs saved to: {output_dir}/")
    print("=" * 70)
    
    return all_results, df_omni_corr, df_cross_corr, df_anom

In [2]:
# ============================================================
# USAGE
# ============================================================

if __name__ == "__main__":
    """
    USAGE:
    ------
    # Save your DataFrame:
    df.to_csv('nmdb_omni_data.csv')
    
    # Quick monthly test:
    results, omni_corr, cross_corr, anomalies = run_full_analysis(
        'nmdb_omni_data.csv', cycles=[24], freq='M',
        compute_or=False, output_dir='./test_monthly')
    
    # Weekly with OR:
    results, omni_corr, cross_corr, anomalies = run_full_analysis(
        'nmdb_omni_data.csv', cycles=[24, 25], freq='W',
        compute_or=True, output_dir='./output_weekly')
    
    # Daily (FR only - OR is too slow for ~4000 nodes):
    results, omni_corr, cross_corr, anomalies = run_full_analysis(
        'nmdb_omni_data.csv', cycles=[24, 25], freq='D',
        compute_or=False, output_dir='./output_daily')
    """
    
    DATA_PATH = 'All_Data.csv'
    
    results, omni_corr, cross_corr, anomalies = run_full_analysis(
        DATA_PATH,
        cr_stations=['JUNG', 'MOSC'],
        cycles=[24, 25],
        freq='D',               # "M", "W", or "D" Montly, Weekly or Daily resolution
        compute_or=False,        # FR only for daily, OR is very heavy....
        compute_rf=False,
        alpha=0.5,
        sigma_anomaly=1.0,
        output_dir='./output_daily_V2'
    )

LOCAL CURVATURE ANALYSIS OF COSMIC RAY TIME SERIES (v2)

[1] Loading data...
Data loaded: 22646 rows, 9 columns
Date range: 1964-01-01 00:00:00 to 2025-12-31 00:00:00
Cycles present: [np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]

Missing data (%):
  JUNG                          : 0.3%
  MOSC                          : 1.6%
  B                             : 11.6%
  Temp                          : 17.0%
  Density                       : 14.7%
  Speed                         : 11.5%
  SSN                           : 0.0%
  Dst                           : 0.0%
  Cycle                         : 0.0%

Cycles to analyze: [24, 25]
Resampling frequency: D
Stations: ['JUNG', 'MOSC']
Compute OR: False | Compute RF: False

[2] Resampling to D...
  Resampled shape: (22646, 9)

  CYCLE 24

  --- JUNG (Cycle 24) ---
  Series length: 4017 (2008-01-02 to 2019-01-01)
  Building VG (4017 nodes)...
  VG: 4017 nodes, 17912 edges (17.2s)
  FR: mean=-5.646, std=9.684 (

# Análisis de Sensibilidad....

In [3]:
"""
Sensitivity Analysis: FR Curvature Anomaly Detection vs FEID Catalog
=====================================================================
Takes pre-computed curvature results + FEID catalog and evaluates
detection performance at multiple sigma thresholds.

For cycle 25 (FEID only reaches Apr 2020), uses Dst proxy as ground truth.

Usage:
    # After running curvature_analysis_v2.py:
    from sensitivity_analysis import run_sensitivity
    run_sensitivity(results, 'FEID_20210101.csv', 'nmdb_omni_data.csv',
                    output_dir='./sensitivity')
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13
})


# ============================================================
# 1. LOAD FEID CATALOG
# ============================================================

def load_feid(filepath, date_min='2008-01-01', date_max='2025-12-31'):
    """
    Load FEID catalog and extract FD events.
    Supports both formats:
      - XLSX with named columns (Date, Time, OType, VMax, HMax, MagnM, DstMin, ...)
      - CSV without header (legacy format from old FEID export)
    
    Type flags (OType):
        1 = confirmed FD (CME-related)
        3 = confirmed (filament eruption)
        9 = automated/small amplitude
    
    Key columns used:
        Date/Time   : event onset
        OType       : event type/quality flag
        VMax        : maximum solar wind speed (km/s)
        HMax        : maximum IMF magnitude (nT)
        MagnM       : FD magnitude (%, GSM method, 10 GV rigidity)
        DstMin      : minimum Dst index during event (nT)
        Kpmax       : maximum Kp index during event
    """
    ext = os.path.splitext(filepath)[1].lower()
    
    # --- XLSX format (named columns) ---
    if ext in ['.xlsx', '.xls']:
        df = pd.read_excel(filepath)
        
        # Helper: replace sentinel values and convert to float
        sentinel_values = [-999, -99, -99.9, -99.99, 9999, -1]
        def clean_col(series):
            s = pd.to_numeric(series, errors='coerce')
            s[s.isin(sentinel_values)] = np.nan
            return s
        
        catalog = pd.DataFrame({
            'date': pd.to_datetime(df['Date'], errors='coerce'),
            'time': df['Time'],
            'type': df['OType'],
            'sw_speed': clean_col(df['VMax']),
            'B_max': clean_col(df['HMax']),
            'magnitude': pd.to_numeric(df['MagnM'], errors='coerce'),
            'dst_min': clean_col(df['DstMin']),
            'kp_max': clean_col(df['Kpmax']),
        })
        
        # MagnM can be 0 or negative for non-events; set to NaN
        catalog.loc[catalog['magnitude'] <= 0, 'magnitude'] = np.nan
    
    # --- CSV format (no header, legacy) ---
    else:
        df = pd.read_csv(filepath, header=None,
                         na_values=['NULL', '-999', '-99', '-99.9'],
                         low_memory=False)
        
        catalog = pd.DataFrame({
            'date': pd.to_datetime(df[0], errors='coerce'),
            'time': df[1],
            'type': df[2],
            'sw_speed': pd.to_numeric(df[8], errors='coerce'),
            'B_max': pd.to_numeric(df[10], errors='coerce'),
            'magnitude': pd.to_numeric(df[12], errors='coerce'),
            'dst_min': pd.NA,
            'kp_max': pd.NA,
        })
    
    # Filter date range and valid magnitude
    catalog = catalog[catalog['date'].notna()].copy()
    mask = (catalog['date'] >= date_min) & (catalog['date'] <= date_max)
    catalog = catalog[mask].copy()
    catalog = catalog[catalog['magnitude'].notna() & (catalog['magnitude'] > 0)].copy()
    catalog = catalog.sort_values('date').reset_index(drop=True)
    
    print(f"FEID catalog loaded: {len(catalog)} events with valid magnitude")
    print(f"  Date range: {catalog['date'].min().date()} to {catalog['date'].max().date()}")
    print(f"  Magnitude: min={catalog['magnitude'].min():.2f}%, "
          f"max={catalog['magnitude'].max():.2f}%, mean={catalog['magnitude'].mean():.2f}%")
    print(f"  Type distribution: {dict(catalog['type'].value_counts().sort_index())}")
    
    # Summary by magnitude tier
    for threshold in [1, 2, 3, 5]:
        n = (catalog['magnitude'] >= threshold).sum()
        print(f"  Events with magnitude >= {threshold}%: {n}")
    
    return catalog


def load_dst_events(omni_filepath, dst_threshold=-30, date_min='2019-01-01', date_max='2025-12-31'):
    """
    Extract geomagnetic storm events from OMNI data as FD proxy for cycle 25.
    """
    ext = os.path.splitext(omni_filepath)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(omni_filepath, index_col=0, parse_dates=True)
    else:
        df = pd.read_pickle(omni_filepath)
    
    col_map = {'Dst-index, nT': 'Dst'}
    df = df.rename(columns=col_map)
    
    if 'Dst' not in df.columns:
        print("WARNING: Dst column not found")
        return pd.DataFrame()
    
    mask = (df.index >= date_min) & (df.index <= date_max)
    dst = df.loc[mask, 'Dst'].dropna()
    
    # Find days where Dst < threshold
    storm_days = dst[dst < dst_threshold]
    
    # Group consecutive storm days into events (keep the minimum Dst day)
    events = []
    if len(storm_days) > 0:
        dates = storm_days.index.to_series()
        gaps = dates.diff() > pd.Timedelta(days=2)
        groups = gaps.cumsum()
        
        for _, group in storm_days.groupby(groups):
            min_idx = group.idxmin()
            events.append({
                'date': min_idx,
                'magnitude': abs(group.min()),  # Use |Dst| as proxy magnitude
                'type': 'Dst_proxy',
                'dst_min': group.min()
            })
    
    catalog = pd.DataFrame(events)
    if not catalog.empty:
        catalog['date'] = pd.to_datetime(catalog['date'])
    
    print(f"Dst proxy events (Dst < {dst_threshold} nT): {len(catalog)} events")
    
    return catalog


# ============================================================
# 2. MATCHING ALGORITHM
# ============================================================

def match_anomalies_to_catalog(anomaly_dates, anomaly_values, catalog_dates, 
                                window_days=2):
    """
    Match detected anomalies to catalog events within a time window.
    
    Returns:
        true_positives: anomalies that match a catalog event
        false_positives: anomalies with no matching catalog event
        false_negatives: catalog events with no matching anomaly
        matches: list of (anomaly_date, catalog_date, time_diff_days)
    """
    matches = []
    matched_catalog = set()
    matched_anomaly = set()
    
    for i, a_date in enumerate(anomaly_dates):
        best_match = None
        best_diff = float('inf')
        
        for j, c_date in enumerate(catalog_dates):
            diff = abs((a_date - c_date).total_seconds()) / 86400  # days
            if diff <= window_days and diff < best_diff:
                best_diff = diff
                best_match = j
        
        if best_match is not None:
            matches.append({
                'anomaly_date': a_date,
                'anomaly_value': anomaly_values[i],
                'catalog_date': catalog_dates[best_match],
                'time_diff_days': best_diff
            })
            matched_catalog.add(best_match)
            matched_anomaly.add(i)
    
    tp = len(matched_anomaly)
    fp = len(anomaly_dates) - tp
    fn = len(catalog_dates) - len(matched_catalog)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'TP': tp, 'FP': fp, 'FN': fn,
        'precision': precision,
        'recall': recall,
        'F1': f1,
        'matches': matches
    }


# ============================================================
# 3. SENSITIVITY ANALYSIS
# ============================================================

def sensitivity_analysis(fr_series, dates, catalog, 
                          sigma_range=np.arange(1.0, 4.1, 0.25),
                          window_days=2,
                          min_magnitude=0.0):
    """
    Run sensitivity analysis across multiple sigma thresholds.
    
    Parameters
    ----------
    fr_series : np.array
        Forman-Ricci curvature values (one per time point)
    dates : pd.DatetimeIndex
    catalog : pd.DataFrame
        FEID catalog with 'date' and 'magnitude' columns
    sigma_range : array
        Sigma values to test
    window_days : int
        Matching window in days
    min_magnitude : float
        Minimum FD magnitude to consider from catalog
    
    Returns
    -------
    results : pd.DataFrame
        Performance metrics per sigma threshold
    """
    # Filter catalog by magnitude
    cat = catalog[catalog['magnitude'] >= min_magnitude].copy()
    catalog_dates = cat['date'].values.astype('datetime64[ns]')
    catalog_dates_list = [pd.Timestamp(d) for d in catalog_dates]
    
    # Filter to overlap period
    date_min = dates.min()
    date_max = dates.max()
    cat_mask = [(d >= date_min) & (d <= date_max) for d in catalog_dates_list]
    catalog_dates_filtered = [d for d, m in zip(catalog_dates_list, cat_mask) if m]
    
    if len(catalog_dates_filtered) == 0:
        print("  WARNING: No catalog events overlap with curvature dates")
        return pd.DataFrame()
    
    mean_fr = np.nanmean(fr_series)
    std_fr = np.nanstd(fr_series)
    
    records = []
    
    for sigma in sigma_range:
        threshold = mean_fr - sigma * std_fr
        
        # Detect anomalies
        anomaly_mask = fr_series < threshold
        anomaly_dates = dates[anomaly_mask]
        anomaly_values = fr_series[anomaly_mask]
        
        if len(anomaly_dates) == 0:
            records.append({
                'sigma': sigma,
                'threshold': threshold,
                'n_anomalies': 0,
                'n_catalog': len(catalog_dates_filtered),
                'TP': 0, 'FP': 0, 'FN': len(catalog_dates_filtered),
                'precision': 0, 'recall': 0, 'F1': 0
            })
            continue
        
        # Match
        result = match_anomalies_to_catalog(
            list(anomaly_dates), list(anomaly_values),
            catalog_dates_filtered, window_days=window_days
        )
        
        records.append({
            'sigma': sigma,
            'threshold': threshold,
            'n_anomalies': len(anomaly_dates),
            'n_catalog': len(catalog_dates_filtered),
            **{k: v for k, v in result.items() if k != 'matches'}
        })
    
    return pd.DataFrame(records)


# ============================================================
# 4. VISUALIZATION
# ============================================================

def plot_sensitivity_curves(results_dict, save_prefix='fig'):
    """
    Plot precision, recall, F1 vs sigma for all stations/cycles in a 2x2 grid.
    """
    n_plots = len(results_dict)
    ncols = 2
    nrows = (n_plots + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = axes.flatten()
    
    fig.suptitle('Sensitivity Analysis: FR Anomaly Detection vs FEID Catalog',
                 fontsize=14, fontweight='bold', y=0.98)
    
    for idx, (label, df) in enumerate(results_dict.items()):
        ax = axes[idx]
        
        ax.plot(df['sigma'], df['precision'], 'b-o', markersize=4, label='Precision', linewidth=1.5)
        ax.plot(df['sigma'], df['recall'], 'r-s', markersize=4, label='Recall', linewidth=1.5)
        ax.plot(df['sigma'], df['F1'], 'g-^', markersize=5, label='F1 Score', linewidth=2)
        
        # Mark optimal F1
        if len(df) > 0 and df['F1'].max() > 0:
            best_idx = df['F1'].idxmax()
            best_sigma = df.loc[best_idx, 'sigma']
            best_f1 = df.loc[best_idx, 'F1']
            ax.axvline(best_sigma, color='green', linestyle='--', alpha=0.5)
            ax.annotate(f'Best: σ={best_sigma:.2f} F1={best_f1:.3f}',
                       xy=(best_sigma, best_f1), fontsize=9,
                       xytext=(best_sigma + 0.5, best_f1 + 0.15),
                       arrowprops=dict(arrowstyle='->', color='black', linewidth=2))
        
        ax.set_xlabel('Sigma threshold (σ)')
        ax.set_ylabel('Score')
        ax.set_title(label)
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0.8, 4.2)
        ax.set_ylim(-0.02, 1.02)
    
    # Hide unused subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_sensitivity_curves.png')
    plt.close()
    print(f"  Saved: {save_prefix}_sensitivity_curves.png")


def plot_sensitivity_by_magnitude(results_by_mag, station_label, save_prefix='fig'):
    """
    Plot F1 curves for different minimum FD magnitude thresholds.
    """
    fig, ax = plt.subplots(figsize=(9, 6))
    fig.suptitle(f'F1 Score vs σ Threshold by FD Magnitude ({station_label})',
                 fontsize=13, fontweight='bold')
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    for idx, (mag_label, df) in enumerate(results_by_mag.items()):
        if len(df) > 0:
            ax.plot(df['sigma'], df['F1'], '-o', color=colors[idx % len(colors)],
                    markersize=4, linewidth=1.5, label=mag_label)
    
    ax.set_xlabel('Sigma threshold (σ)')
    ax.set_ylabel('F1 Score')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.8, 4.2)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_sensitivity_by_magnitude_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_sensitivity_by_magnitude_{station_label}.png")


def plot_detection_summary(fr_series, dates, catalog, optimal_sigma, 
                            station_label, window_days=2, save_prefix='fig'):
    """
    Plot the final detection result at optimal sigma, showing matches.
    """
    mean_fr = np.nanmean(fr_series)
    std_fr = np.nanstd(fr_series)
    threshold = mean_fr - optimal_sigma * std_fr
    
    anomaly_mask = fr_series < threshold
    anomaly_dates = dates[anomaly_mask]
    anomaly_values = fr_series[anomaly_mask]
    
    # Filter catalog to date range
    cat = catalog[(catalog['date'] >= dates.min()) & (catalog['date'] <= dates.max())]
    catalog_dates = [pd.Timestamp(d) for d in cat['date'].values]
    
    result = match_anomalies_to_catalog(
        list(anomaly_dates), list(anomaly_values),
        catalog_dates, window_days=window_days
    )
    
    fig = plt.figure(figsize=(16, 13))
    gs = fig.add_gridspec(3, 1, hspace=0.35)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1], sharex=ax0)  # shares x with panel a
    ax2 = fig.add_subplot(gs[2])               # independent x-axis
    axes = [ax0, ax1, ax2]
    
    fig.suptitle(f'Detection at Optimal σ={optimal_sigma:.2f} ({station_label})\n'
                 f'TP={result["TP"]}, FP={result["FP"]}, FN={result["FN"]} | '
                 f'Precision={result["precision"]:.3f}, Recall={result["recall"]:.3f}, '
                 f'F1={result["F1"]:.3f}',
                 fontsize=12, fontweight='bold', y=0.98)
    
    # Panel a: FR curvature with detections
    axes[0].plot(dates, fr_series, 'steelblue', linewidth=0.6, alpha=0.8)
    axes[0].axhline(threshold, color='red', linestyle='--', linewidth=1,
                     label=f'Threshold (μ-{optimal_sigma:.2f}σ = {threshold:.1f})')
    
    # Color-code anomalies
    matched_dates = set(pd.Timestamp(m['anomaly_date']) for m in result['matches'])
    
    for i, (ad, av) in enumerate(zip(anomaly_dates, anomaly_values)):
        if pd.Timestamp(ad) in matched_dates:
            axes[0].scatter(ad, av, c='green', s=20, zorder=5, alpha=0.7)
        else:
            axes[0].scatter(ad, av, c='red', s=15, zorder=5, alpha=0.5)
    
    axes[0].scatter([], [], c='green', s=30, label=f'True Positive ({result["TP"]})')
    axes[0].scatter([], [], c='red', s=30, label=f'False Positive ({result["FP"]})')
    axes[0].set_ylabel('FR Curvature')
    axes[0].set_title('(a) Forman-Ricci Curvature with Detections', loc='left')
    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Panel b: FEID catalog events
    if not cat.empty:
        axes[1].bar(cat['date'], cat['magnitude'], width=3, color='coral', alpha=0.7)
        
        # Mark undetected events
        detected_cat = set()
        for m in result['matches']:
            detected_cat.add(pd.Timestamp(m['catalog_date']))
        
        for _, row in cat.iterrows():
            if pd.Timestamp(row['date']) not in detected_cat:
                axes[1].scatter(row['date'], row['magnitude'], 
                              c='black', marker='x', s=40, zorder=5)
        
        axes[1].scatter([], [], c='black', marker='x', s=40, 
                       label=f'Missed FD ({result["FN"]})')
    
    axes[1].set_ylabel('FD Magnitude (%)')
    axes[1].set_title('(b) FEID Catalog Events', loc='left')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    # Panel c: Time difference histogram for matches
    if result['matches']:
        diffs = [m['time_diff_days'] for m in result['matches']]
        axes[2].hist(diffs, bins=np.arange(0, window_days + 0.2, 0.1),
                     color='steelblue', alpha=0.7, edgecolor='white')
        axes[2].set_xlabel('Time difference (days)')
        axes[2].set_ylabel('Count')
        axes[2].set_title(f'(c) Matching Time Offset (mean={np.mean(diffs):.2f} days, '
                          f'median={np.median(diffs):.2f} days)', loc='left')
        axes[2].axvline(np.mean(diffs), color='red', linestyle='--', label=f'Mean = {np.mean(diffs):.2f} d')
        axes[2].axvline(np.median(diffs), color='orange', linestyle='--', label=f'Median = {np.median(diffs):.2f} d')
        axes[2].set_xlim(-0.05, window_days + 0.1)
        axes[2].legend(fontsize=8)
    else:
        axes[2].text(0.5, 0.5, 'No matches', transform=axes[2].transAxes,
                    ha='center', va='center', fontsize=14)
    
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_detection_summary_{station_label}.png')
    plt.close()
    print(f"  Saved: {save_prefix}_detection_summary_{station_label}.png")


# ============================================================
# 5. MAIN FUNCTION
# ============================================================

def run_sensitivity(all_results, feid_path, omni_path=None,
                     cycles_feid=[24], cycles_dst=[25],
                     sigma_range=np.arange(1.0, 4.1, 0.25),
                     magnitude_thresholds=[0, 1, 2, 3, 5],
                     window_days=2, dst_threshold=-30,
                     output_dir='./sensitivity'):
    """
    Run complete sensitivity analysis.
    
    Parameters
    ----------
    all_results : dict
        {cycle: {station: result_dict}} from curvature_analysis_v2
    feid_path : str
        Path to FEID CSV
    omni_path : str or None
        Path to OMNI data (for Dst proxy in cycle 25)
    cycles_feid : list
        Cycles to validate against FEID
    cycles_dst : list
        Cycles to validate against Dst proxy
    sigma_range : array
    magnitude_thresholds : list
        Minimum FD magnitudes to test
    window_days : int
    dst_threshold : float
        Dst threshold for proxy events
    output_dir : str
    """
    os.makedirs(output_dir, exist_ok=True)
    save_prefix = os.path.join(output_dir, 'sens')
    
    print("=" * 70)
    print("SENSITIVITY ANALYSIS: FR Anomaly Detection")
    print("=" * 70)
    
    # Load FEID
    print("\n[1] Loading FEID catalog...")
    feid = load_feid(feid_path)
    
    # Load Dst proxy if needed
    dst_catalog = None
    if omni_path and cycles_dst:
        print("\n[2] Loading Dst proxy events...")
        dst_catalog = load_dst_events(omni_path, dst_threshold=dst_threshold)
    
    # Run sensitivity per station/cycle
    all_sensitivity = {}
    all_sensitivity_by_mag = {}
    optimal_sigmas = {}
    
    for cycle in list(set(cycles_feid + (cycles_dst or []))):
        if cycle not in all_results:
            print(f"\n  Cycle {cycle}: not in results, skipping")
            continue
        
        # Select catalog
        if cycle in cycles_feid:
            cat = feid.copy()
            cat_source = 'FEID'
        elif dst_catalog is not None:
            cat = dst_catalog.copy()
            cat_source = 'Dst proxy'
        else:
            continue
        
        for station, result in all_results[cycle].items():
            label = f"Cycle {cycle} - {station} ({cat_source})"
            print(f"\n{'='*50}")
            print(f"  {label}")
            print(f"{'='*50}")
            
            fr = result['FR_node']
            dates = result['dates']
            
            # --- Main sensitivity (all FD magnitudes) ---
            print(f"\n  Running sensitivity (all magnitudes, window={window_days}d)...")
            sens = sensitivity_analysis(fr, dates, cat,
                                         sigma_range=sigma_range,
                                         window_days=window_days,
                                         min_magnitude=0)
            all_sensitivity[label] = sens
            
            if len(sens) > 0 and sens['F1'].max() > 0:
                best = sens.loc[sens['F1'].idxmax()]
                optimal_sigmas[label] = best['sigma']
                print(f"  Best σ={best['sigma']:.2f}: "
                      f"F1={best['F1']:.3f}, "
                      f"Precision={best['precision']:.3f}, "
                      f"Recall={best['recall']:.3f}, "
                      f"TP={int(best['TP'])}, FP={int(best['FP'])}, FN={int(best['FN'])}")
            
            # --- Sensitivity by FD magnitude ---
            print(f"\n  Running sensitivity by magnitude tiers...")
            mag_results = {}
            for mag in magnitude_thresholds:
                mag_label = f'FD ≥ {mag}%'
                s = sensitivity_analysis(fr, dates, cat,
                                          sigma_range=sigma_range,
                                          window_days=window_days,
                                          min_magnitude=mag)
                mag_results[mag_label] = s
                
                if len(s) > 0 and s['F1'].max() > 0:
                    b = s.loc[s['F1'].idxmax()]
                    n_cat = int(b['n_catalog'])
                    print(f"    {mag_label} (N={n_cat}): "
                          f"best σ={b['sigma']:.2f}, F1={b['F1']:.3f}")
            
            all_sensitivity_by_mag[f"C{cycle}_{station}"] = mag_results
            
            # Plot magnitude tiers
            plot_sensitivity_by_magnitude(mag_results, f"C{cycle}_{station}",
                                          save_prefix=save_prefix)
    
    # --- Global plots ---
    print("\n\n[3] Generating summary plots...")
    
    # Sensitivity curves
    plot_sensitivity_curves(all_sensitivity, save_prefix=save_prefix)
    
    # Detection summary at optimal sigma for each
    for label, sigma in optimal_sigmas.items():
        # Parse cycle and station from label
        parts = label.split(' - ')
        cycle = int(parts[0].replace('Cycle ', ''))
        station = parts[1].split(' ')[0]
        
        if cycle in all_results and station in all_results[cycle]:
            result = all_results[cycle][station]
            cat = feid if cycle in cycles_feid else dst_catalog
            
            plot_detection_summary(
                result['FR_node'], result['dates'], cat,
                sigma, f"C{cycle}_{station}",
                window_days=window_days, save_prefix=save_prefix
            )
    
    # --- Save results ---
    print("\n[4] Saving results...")
    
    # Summary table
    summary_records = []
    for label, df in all_sensitivity.items():
        if len(df) > 0:
            for _, row in df.iterrows():
                summary_records.append({'analysis': label, **row.to_dict()})
    
    if summary_records:
        df_summary = pd.DataFrame(summary_records)
        df_summary.to_csv(os.path.join(output_dir, 'sensitivity_results.csv'), index=False)
        
        # Print optimal results
        print("\n" + "=" * 70)
        print("OPTIMAL THRESHOLDS (maximizing F1)")
        print("=" * 70)
        
        for label, sigma in optimal_sigmas.items():
            df = all_sensitivity[label]
            best = df.loc[df['F1'].idxmax()]
            print(f"\n  {label}:")
            print(f"    σ = {best['sigma']:.2f} (threshold = {best['threshold']:.1f})")
            print(f"    Anomalies detected: {int(best['n_anomalies'])}")
            print(f"    Catalog events: {int(best['n_catalog'])}")
            print(f"    TP={int(best['TP'])}, FP={int(best['FP'])}, FN={int(best['FN'])}")
            print(f"    Precision = {best['precision']:.4f}")
            print(f"    Recall    = {best['recall']:.4f}")
            print(f"    F1 Score  = {best['F1']:.4f}")
    
    print(f"\nAll outputs saved to: {output_dir}/")
    print("=" * 70)
    
    return all_sensitivity, optimal_sigmas


# ============================================================
# STANDALONE USAGE
# ============================================================

if __name__ == "__main__":
    """
    USAGE:
    
    The FEID XLSX file covers up to Dec 2021, so it can validate both 
    cycle 24 (full) and cycle 25 (partial, 2019-2021) against FEID.
    For cycle 25 after 2021, Dst proxy is used automatically.
    
    After running curvature_analysis_v2.py:
    
        from sensitivity_analysis import run_sensitivity
        
        # Using XLSX (recommended):
        sens, optimal = run_sensitivity(
            results,
            'FEID-2.xlsx',                # XLSX with named columns
            'nmdb_omni_data.csv',         # for Dst proxy (post-2021)
            cycles_feid=[24, 25],         # both cycles against FEID
            cycles_dst=[],                # or [25] to also test Dst proxy
            output_dir='./sensitivity'
        )
        
        # Using old CSV:
        sens, optimal = run_sensitivity(
            results,
            'FEID_20210101.csv',          # CSV without header
            'nmdb_omni_data.csv',
            cycles_feid=[24],
            cycles_dst=[25],
            output_dir='./sensitivity'
        )
    """
    print("Run this module after computing curvature results.")
    print("See docstring for usage examples.")

Run this module after computing curvature results.
See docstring for usage examples.


**Table from:** http://crsb.izmiran.ru/phpmyadmin/tbl_export.php?db=cosray&table=FEID&single_table=true

In [4]:
#from sensitivity_analysis import run_sensitivity

sens, optimal = run_sensitivity(
    results,
    'FEID-2.xlsx',              # XLSX con columnas nombradas
    'All_Data.csv.csv',       # para Dst proxy post-2021
    cycles_feid=[24, 25],       # ambos ciclos contra FEID
    cycles_dst=[],              # vacío si no se quiere Dst proxy
    output_dir='./sensitivity'
)

SENSITIVITY ANALYSIS: FR Anomaly Detection

[1] Loading FEID catalog...
FEID catalog loaded: 1832 events with valid magnitude
  Date range: 2008-01-04 to 2021-12-29
  Magnitude: min=0.10%, max=11.20%, mean=0.94%
  Type distribution: {1: np.int64(182), 2: np.int64(1), 3: np.int64(35), 4: np.int64(5), 5: np.int64(10), 6: np.int64(1), 9: np.int64(1598)}
  Events with magnitude >= 1%: 576
  Events with magnitude >= 2%: 124
  Events with magnitude >= 3%: 47
  Events with magnitude >= 5%: 14

  Cycle 24 - JUNG (FEID)

  Running sensitivity (all magnitudes, window=2d)...
  Best σ=1.00: F1=0.408, Precision=0.958, Recall=0.260, TP=408, FP=18, FN=1164

  Running sensitivity by magnitude tiers...
    FD ≥ 0% (N=1433): best σ=1.00, F1=0.408
    FD ≥ 1% (N=514): best σ=1.00, F1=0.509
    FD ≥ 2% (N=121): best σ=1.25, F1=0.231
    FD ≥ 3% (N=46): best σ=1.50, F1=0.125
    FD ≥ 5% (N=13): best σ=1.50, F1=0.054
  Saved: ./sensitivity/sens_sensitivity_by_magnitude_C24_JUNG.png

  Cycle 24 - MOSC (FEID)